# Installation

In [ ]:
#pip uninstall -y transformers torch torchvision

In [ ]:
#!pip install git+https://github.com/dnth/rag-datakit.git

In [1]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import load_dataset

In [2]:
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-batch10")
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-v2")
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-v3")
dataset = load_dataset("frankwong2001/ssf-train-valid-combi-v1v2v3")
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 10556
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 2639
    })
})

In [3]:
dataset['valid'][0]

{'anchor': "The General Manager/Managing Director/Vice President (Aircraft Maintenance) is responsible for defining the long-term strategic direction to grow the business in line with the organisations overall vision, mission and values. He/She promotes strategic aircraft maintenance programmes for business competitiveness and sets direction for leading aerospace maintenance practices in the organisation. He represents the organisation with customers, investors, and business partners, and holds responsibility for promoting organisational compliance with airworthiness and legislative requirements, fostering a culture of workplace safety and health, and championing leading practices and quality and risk management. He inspires the organisation towards achieving business goals by striving for continuous improvement, driving digital innovation and evaluating the organisation's approach towards a lean and sustainable enterprise. He demonstrates excellent leadership capabilities and builds s

# W&B and Model Configuration

In [4]:
import wandb
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Fetch the WANDB_API_KEY from the environment
wandb_api_key = os.getenv("WAB_API_KEY")

# Log in using the API key
wandb.login(key=wandb_api_key)

model_id = "nomic-ai/modernbert-embed-base"
save_model_path = "./models/nomic-ai/modernbert-embed-base"
wandb.init(project="rag-datakit-finetunes", name="4_nomic-ai/modernbert-embed-base~frankwong2001/ssf-train-valid-combi-v1v2v3")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: frankwong2001 (frankwong2001-cxsanalytics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Training Arguments

In [5]:
args = SentenceTransformerTrainingArguments(
    output_dir=save_model_path,
    num_train_epochs=5,                         # number of epochs
    per_device_train_batch_size=32,             # train batch size
    gradient_accumulation_steps=16,             # for a global batch size of 512
    per_device_eval_batch_size=16,              # evaluation batch size
    warmup_ratio=0.1,                           # warmup ratio
    learning_rate=2e-5,                         # learning rate, 2e-5 is a good value
    lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
    optim="adamw_torch_fused",
    tf32=False,                                 # use tf32 precision
    bf16=True,         
    #fp16=True,                                                  # use bf16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="epoch",                      # evaluate after each epoch
    save_strategy="epoch",                      # save after each epoch
    logging_strategy="epoch",                   # log after each epoch
    save_total_limit=3,                         # save only the last 3 models
    load_best_model_at_end=True,                # load the best model when training ends
    report_to="wandb",
    # gradient_checkpointing=True,              # use fused adamw optimizer
    #use_cache=False                            # disable the use of cache
    # weight_decay=0.01,                             # apply weight decay
    # max_grad_norm=0.5,                           # clip the gradient norm
    # warmup_steps=1500                           # number of warmup steps
    )

In [6]:
model = SentenceTransformer(model_id)
train_loss = MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['valid'],  
    loss=train_loss,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

# Execute Training


In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 204.00 MiB. GPU 0 has a total capacity of 44.45 GiB of which 174.62 MiB is free. Process 2428183 has 44.27 GiB memory in use. Of the allocated memory 41.53 GiB is allocated by PyTorch, and 2.41 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

#  Save & Upload Model

In [ ]:
trainer.save_model()

In [ ]:
import os
wandb.save(os.path.join(save_model_path, "*"))

In [ ]:
wandb.finish()

# Push to Hugging Face

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import Trainer

# Load environment variables from .env file
load_dotenv()

# Fetch the Hugging Face API key from the environment
hf_api_key = os.getenv("HF_TOKEN")

# Log in using the Hugging Face API key
login(token=hf_api_key)

# Assuming you have a Trainer object `trainer`
trainer.model.push_to_hub("frankwong2001/4_modernbert-embed-base", exist_ok=True)
